# 06 — BT Pair Schedule Generation

Generates a pairwise comparison schedule for Bradley-Terry (BT) estimation
from the stratified intersection sample produced by notebook 05.

The BT model requires a *connected* comparison graph across all intersections
so that scores sit on a common scale. This notebook builds that graph with
two types of pairs:

- **Within-stratum pairs** — dense comparisons within each of the 6 strata
- **Bridging pairs** — connect strata to each other via a spanning tree + extra diagonals

**Design details:**
- Small strata (≤20 nodes, i.e. T/VRI = 16) get a full round-robin
- Larger strata get an MST backbone topped up to ≥8 comparisons per node
- 5 spanning-tree bridges (15 pairs each) guarantee global connectivity
- 3 extra diagonal bridges (10 pairs each) add robustness against cross-arm score drift

**Input:**
- `data/processed/sampled_legs_{IMAGE_MODE}.csv` — stratified sample from notebook 05

**Outputs:**
- `data/processed/bt_pairs.csv` — one row per comparison pair, ready for survey tool
- `data/processed/bt_pairs_summary.csv` — per-stratum connectivity and balance report

**Depends on:** notebook 05 must be run first.

In [1]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import itertools
import random

# Base project directory — same convention as other notebooks
PROJECT_DIR = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections"

# --- Image mode ---
# Must match the IMAGE_MODE used when running notebooks 02, 04, and 05.
# "panorama"   : one 360° image per location
# "directional": four shots per location
IMAGE_MODE = "directional"  # change to "directional" to process four-direction photos

# Input: stratified leg sample from notebook 05 — filename reflects IMAGE_MODE
INPUT_PATH = os.path.join(PROJECT_DIR, "data", "processed",
                          f"sampled_legs_{IMAGE_MODE}.csv")

# Output: single merged design file — one row per survey task, with all columns
# needed for (1) the survey tool, (2) vault lookup, and (3) BT model fitting.
# Replaces the old separate bt_pairs.csv and pairwise_design.csv.
OUTPUT_DESIGN_PATH = os.path.join(PROJECT_DIR, "data", "processed", "survey_design.csv")

# Diagnostic summary: per-stratum connectivity stats (kept separate — small, useful for QC)
SUMMARY_PATH = os.path.join(PROJECT_DIR, "data", "processed", "bt_pairs_summary.csv")

# Canonical deliverable: also copied to output/ for handoff
OUTPUT_DIR = os.path.join(PROJECT_DIR, "output")

# Reproducibility seed — used for all random operations
RANDOM_SEED = 42

# Map logical names to actual column names in sampled_legs.csv.
COLUMN_MAP = {
    "intersection_id": "intersection_id",
    "image_path":      "photo_filepath",
    "dim_type":        "dim_type",
    "dim_priority":    "dim_priority",
}

# Target minimum comparisons per node within each stratum.
# Strata with <=20 nodes use a full round-robin instead (ignores this setting).
TARGET_COMP_PER_NODE = 7

# Pairs per stratum link for the spanning-tree bridges
BRIDGE_PAIRS_PER_LINK = 15

# Pairs per link for the extra diagonal bridges
EXTRA_BRIDGE_PAIRS = 10

# Spanning tree of 6 strata: exactly 5 links needed for global connectivity.
BRIDGE_SPANNING_TREE = [
    ("4+_VRI",           "T_VRI"),
    ("4+_geen_voorrang", "T_geen_voorrang"),
    ("4+_voorrang",      "T_voorrang"),
    ("4+_VRI",           "4+_geen_voorrang"),
    ("4+_geen_voorrang", "4+_voorrang"),
]

# Extra diagonal bridges — cross both arm (4+ vs T) and priority simultaneously.
EXTRA_BRIDGE_LINKS = [
    ("4+_VRI",           "T_geen_voorrang"),
    ("4+_geen_voorrang", "T_VRI"),
    ("4+_voorrang",      "T_VRI"),
]

# -- Rater assignment ----------------------------------------------------------
N_RATERS = 16

# Time budget: 25 min × 2 pairs/min = 50 pairs per rater.
MINUTES_PER_RATER = 25
PAIRS_PER_MINUTE  = 2
PAIRS_PER_RATER   = int(MINUTES_PER_RATER * PAIRS_PER_MINUTE)  # 50

# Anchor pairs: shown to ALL raters for inter-rater reliability checks.
N_ANCHOR_PAIRS = 10

# Replication: how many raters see each non-anchor pair.
BRIDGE_REPLICATION = 3
RATER_REPLICATION  = 2

# Output directory for per-rater schedule CSVs (kept for distributing to individual raters)
RATER_SCHEDULE_DIR = os.path.join(PROJECT_DIR, "data", "processed", "rater_schedules")

print(f"IMAGE_MODE   : {IMAGE_MODE}")
print(f"INPUT        : {os.path.basename(INPUT_PATH)}")
print(f"OUTPUT       : {os.path.basename(OUTPUT_DESIGN_PATH)}")

IMAGE_MODE   : directional
INPUT        : sampled_legs_directional.csv
OUTPUT       : survey_design.csv


## 1. Load data

In [2]:
rng = random.Random(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Load the leg file — one row per intersection (random leg selected in notebook 05)
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df):,} rows from sampled_legs.csv")

id_col  = COLUMN_MAP["intersection_id"]
img_col = COLUMN_MAP["image_path"]

# Build a combined stratum label from dim_type + dim_priority.
# This matches the 6-stratum structure designed in notebook 03.
df["_stratum"] = (
    df[COLUMN_MAP["dim_type"]].astype(str).str.strip()
    + "_"
    + df[COLUMN_MAP["dim_priority"]].astype(str).str.strip()
)

# Include intensity_wvk if notebook 04 produced it (USE_INTENSITY=True).
if "intensity_wvk" not in df.columns:
    df["intensity_wvk"] = np.nan
    print("Note: intensity_wvk not found — run notebook 04 with USE_INTENSITY=True to populate it.")

# sampled_legs.csv already contains one row per intersection.
inter_df = (
    df[[id_col, img_col, "_stratum",
        COLUMN_MAP["dim_type"], COLUMN_MAP["dim_priority"], "intensity_wvk"]]
    .reset_index(drop=True)
)

print(f"\n{len(inter_df)} unique intersections across strata:")
for s, g in inter_df.groupby("_stratum"):
    print(f"  {s}: {len(g)} intersections")

# --- Attribute columns for DCE-style balanced pairing ---

# Intensity quartile: bins each intersection into one of 4 equal-frequency groups.
# Fill missing values with the overall median so sparse intersections don't skew edges.
intensity_filled = inter_df["intensity_wvk"].fillna(inter_df["intensity_wvk"].median())
inter_df["intensity_q"] = pd.qcut(
    intensity_filled, q=4, labels=False, duplicates="drop"
).astype(int)
print(f"\nIntensity quartile distribution (0=lowest, 3=highest traffic):")
print(inter_df["intensity_q"].value_counts().sort_index().to_string())

# is_centrum: passed from nb03 via nb05 if USE_CENTRUM=True.
# Falls back to False (all non-centrum) if the column is absent, so the design
# degrades gracefully to intensity-only balancing without any code change.
if "is_centrum" in df.columns:
    inter_df["is_centrum"] = df["is_centrum"].fillna(False).astype(bool)
    print(f"\nis_centrum: {inter_df['is_centrum'].sum()} centrum intersections "
          f"({inter_df['is_centrum'].mean()*100:.1f}%)")
else:
    inter_df["is_centrum"] = False
    print("\nNote: is_centrum not in sampled_legs.csv — "
          "re-run nb03 and nb05 with USE_CENTRUM=True to enable geographic balancing.")

# Build metadata index now so helper functions can access it during pair generation.
# (Also rebuilt at the top of the output cell for clarity; the second assignment is harmless.)
meta = inter_df.set_index(id_col)

Loaded 368 rows from sampled_legs.csv

368 unique intersections across strata:
  4+_VRI: 70 intersections
  4+_geen_voorrang: 70 intersections
  4+_voorrang: 70 intersections
  T_VRI: 18 intersections
  T_geen_voorrang: 70 intersections
  T_voorrang: 70 intersections

Intensity quartile distribution (0=lowest, 3=highest traffic):
intensity_q
0     95
1    104
2     79
3     90

is_centrum: 51 centrum intersections (13.9%)


## 2. Helper functions

In [3]:
def build_mst_backbone(node_ids):
    """Build a simple path 0→1→…→n-1 as a cheap spanning tree.
    Guarantees within-stratum connectivity before any random fill.
    """
    return [(node_ids[i], node_ids[i + 1]) for i in range(len(node_ids) - 1)]


def full_round_robin(node_ids):
    """All possible pairs — cheap enough for small strata (e.g. T/VRI = 17 nodes)."""
    return list(itertools.combinations(node_ids, 2))


def fill_to_target_balanced(node_ids, existing_pairs, target, meta, rng):
    """Fill pairs until every node has >= target comparisons.

    Candidate pairs are sorted by attribute diversity before the greedy fill:
    pairs that differ on intensity quartile and centrum status are preferred,
    ensuring the within-stratum candidate pool contains informative comparisons
    rather than many near-identical intersections.
    """
    all_possible = list(itertools.combinations(node_ids, 2))

    # Score: higher = more diverse in key attributes.
    # Weight intensity range (0–6) more than centrum flag (0–1).
    def attr_div(a, b):
        iq = abs(int(meta.loc[a, "intensity_q"]) - int(meta.loc[b, "intensity_q"]))
        ct = int(bool(meta.loc[a, "is_centrum"]) != bool(meta.loc[b, "is_centrum"]))
        return iq * 2 + ct

    # Sort descending by diversity; break ties with a random key for reproducibility
    scored = [(attr_div(a, b), rng.random(), a, b) for a, b in all_possible]
    scored.sort(key=lambda x: (-x[0], x[1]))
    all_possible = [(a, b) for _, _, a, b in scored]

    existing_set = set(map(frozenset, existing_pairs))
    comp_count = {n: 0 for n in node_ids}
    for a, b in existing_pairs:
        comp_count[a] += 1
        comp_count[b] += 1

    new_pairs = []
    for a, b in all_possible:
        if frozenset({a, b}) in existing_set:
            continue
        if comp_count[a] >= target and comp_count[b] >= target:
            continue
        new_pairs.append((a, b))
        existing_set.add(frozenset({a, b}))
        comp_count[a] += 1
        comp_count[b] += 1
        if all(v >= target for v in comp_count.values()):
            break
    return new_pairs


def sample_bridge_pairs_balanced(nodes_a, nodes_b, n, meta, rng):
    """Sample n cross-stratum pairs, preferring those with diverse intensity and centrum attributes.

    Two-pass selection:
    1. Ensure every node in both strata appears at least once (coverage guarantee).
    2. Fill remaining slots from the pool sorted by attribute diversity descending.
    """
    pool = list(itertools.product(nodes_a, nodes_b))

    def attr_div(a, b):
        iq = abs(int(meta.loc[a, "intensity_q"]) - int(meta.loc[b, "intensity_q"]))
        ct = int(bool(meta.loc[a, "is_centrum"]) != bool(meta.loc[b, "is_centrum"]))
        return iq + ct

    # Sort by diversity descending; break ties randomly for reproducibility
    scored = [(attr_div(a, b), rng.random(), a, b) for a, b in pool]
    scored.sort(key=lambda x: (-x[0], x[1]))
    pool_sorted = [(a, b) for _, _, a, b in scored]

    # Pass 1: include any pair that introduces a new node on either side
    selected, seen_a, seen_b = [], set(), set()
    for a, b in pool_sorted:
        if a not in seen_a or b not in seen_b:
            selected.append((a, b))
            seen_a.add(a)
            seen_b.add(b)
        if len(selected) >= n:
            return selected[:n]

    # Pass 2: fill to n from the remaining highest-diversity pairs
    selected_set = set(selected)
    for a, b in pool_sorted:
        if (a, b) not in selected_set:
            selected.append((a, b))
            selected_set.add((a, b))
        if len(selected) >= n:
            break
    return selected[:n]


def select_space_filling_within(within_pairs, meta, n_select):
    """Select n_select within-stratum pairs using round-robin over attribute-difference bins.

    Bins each pair by (|intensity_q_a - intensity_q_b|, is_centrum_a != is_centrum_b).
    Draws from bins in round-robin order starting from the highest-diversity bin, so the
    selected subset covers the full attribute-difference space as evenly as possible.

    Returns (selected_pairs, bin_report) where bin_report maps
    (iq_diff, centrum_diff) -> {"selected": int, "available": int}.
    """
    from collections import defaultdict

    # Bin each pair by its attribute-difference signature
    bins = defaultdict(list)
    for a, b in within_pairs:
        iq = abs(int(meta.loc[a, "intensity_q"]) - int(meta.loc[b, "intensity_q"]))
        ct = int(bool(meta.loc[a, "is_centrum"]) != bool(meta.loc[b, "is_centrum"]))
        bins[(iq, ct)].append((a, b))

    # Record original bin sizes before drawing
    bin_sizes = {k: len(v) for k, v in bins.items()}

    # Round-robin draw, highest-diversity bin first: (3,1) > (3,0) > (2,1) > …
    bin_keys = sorted(bins.keys(), reverse=True)
    selected = []
    while len(selected) < n_select:
        any_added = False
        for key in bin_keys:
            if bins[key] and len(selected) < n_select:
                selected.append(bins[key].pop(0))
                any_added = True
        if not any_added:
            break  # all bins exhausted

    # Build coverage report
    bin_report = {}
    for key in sorted(bin_sizes.keys(), reverse=True):
        drawn = bin_sizes[key] - len(bins.get(key, []))
        bin_report[key] = {"selected": drawn, "available": bin_sizes[key]}

    return selected, bin_report


print("Helper functions defined.")

Helper functions defined.


## 3. Within-stratum pairs

In [4]:
all_pairs = []       # accumulates all (id_a, id_b) tuples
stratum_nodes = {}   # maps stratum label -> list of intersection IDs

for stratum, group in inter_df.groupby("_stratum"):
    nodes = group[id_col].tolist()
    stratum_nodes[stratum] = nodes

    if len(nodes) <= 20:
        # Full round-robin is cheap for small strata (T/VRI = 17 nodes gives 136 pairs)
        pairs = full_round_robin(nodes)
        print(f"  {stratum}: full round-robin -> {len(pairs)} pairs ({len(nodes)} nodes)")
    else:
        # For larger strata: MST backbone guarantees connectivity, then attribute-balanced
        # fill adds pairs that differ on intensity quartile and centrum status first.
        backbone = build_mst_backbone(nodes)
        extra    = fill_to_target_balanced(nodes, backbone, TARGET_COMP_PER_NODE, meta, rng)
        pairs    = backbone + extra
        print(f"  {stratum}: MST ({len(backbone)}) + balanced fill ({len(extra)}) -> {len(pairs)} pairs ({len(nodes)} nodes)")

    all_pairs.extend(pairs)

print(f"\nWithin-stratum candidate pairs total: {len(all_pairs)}")

  4+_VRI: MST (69) + balanced fill (296) -> 365 pairs (70 nodes)


  4+_geen_voorrang: MST (69) + balanced fill (309) -> 378 pairs (70 nodes)


  4+_voorrang: MST (69) + balanced fill (321) -> 390 pairs (70 nodes)
  T_VRI: full round-robin -> 153 pairs (18 nodes)


  T_geen_voorrang: MST (69) + balanced fill (319) -> 388 pairs (70 nodes)


  T_voorrang: MST (69) + balanced fill (314) -> 383 pairs (70 nodes)

Within-stratum candidate pairs total: 2057


## 4. Bridging pairs

Without bridging, the 6 strata form 6 disconnected components and BT scores
cannot be compared across strata. The spanning tree links guarantee connectivity;
the extra diagonal links add robustness.

In [5]:
print("Spanning-tree bridges (minimum required for global connectivity):")
for sa, sb in BRIDGE_SPANNING_TREE:
    if sa not in stratum_nodes or sb not in stratum_nodes:
        print(f"  WARNING: stratum {sa!r} or {sb!r} not found in data - skipping")
        continue
    bridge = sample_bridge_pairs_balanced(
        stratum_nodes[sa], stratum_nodes[sb], BRIDGE_PAIRS_PER_LINK, meta, rng
    )
    print(f"  {sa} <-> {sb}: {len(bridge)} pairs")
    all_pairs.extend(bridge)

print("\nExtra diagonal bridges (robustness against cross-arm score drift):")
for sa, sb in EXTRA_BRIDGE_LINKS:
    if sa not in stratum_nodes or sb not in stratum_nodes:
        print(f"  WARNING: stratum {sa!r} or {sb!r} not found in data - skipping")
        continue
    bridge = sample_bridge_pairs_balanced(
        stratum_nodes[sa], stratum_nodes[sb], EXTRA_BRIDGE_PAIRS, meta, rng
    )
    print(f"  {sa} <-> {sb}: {len(bridge)} pairs")
    all_pairs.extend(bridge)

Spanning-tree bridges (minimum required for global connectivity):
  4+_VRI <-> T_VRI: 15 pairs


  4+_geen_voorrang <-> T_geen_voorrang: 15 pairs


  4+_voorrang <-> T_voorrang: 15 pairs


  4+_VRI <-> 4+_geen_voorrang: 15 pairs
  4+_geen_voorrang <-> 4+_voorrang: 15 pairs

Extra diagonal bridges (robustness against cross-arm score drift):


  4+_VRI <-> T_geen_voorrang: 10 pairs
  4+_geen_voorrang <-> T_VRI: 10 pairs
  4+_voorrang <-> T_VRI: 10 pairs


## 5. Deduplicate and verify connectivity

The MST fill and bridging steps can produce the same pair independently.
Deduplicate first, then verify the graph is connected — BT estimation
requires a single connected component.

In [6]:
# Remove duplicate pairs produced by overlapping MST and bridging steps
seen = set()
deduped = []
for a, b in all_pairs:
    fs = frozenset({a, b})
    if fs not in seen:
        seen.add(fs)
        deduped.append((a, b))
print(f"Total unique pairs after deduplication: {len(deduped)}")

# Build the comparison graph and check connectivity with networkx.
# A disconnected graph means at least two strata have no bridging pairs between them,
# which makes cross-stratum BT scores non-comparable.
all_nodes = inter_df[id_col].tolist()
G = nx.Graph()
G.add_nodes_from(all_nodes)
G.add_edges_from(deduped)
connected = nx.is_connected(G)
print(f"Graph connected: {connected}")

if not connected:
    components = list(nx.connected_components(G))
    print(f"\nWARNING: {len(components)} disconnected components found.")
    print("Add more bridging pairs to connect the following isolated groups:")
    for i, comp in enumerate(components):
        strata_in = inter_df[inter_df[id_col].isin(comp)]["_stratum"].unique()
        sample_ids = list(comp)[:5]
        print(f"  Component {i+1} ({len(comp)} nodes, strata: {list(strata_in)}): {sample_ids} ...")
else:
    print("All intersections reachable from each other — BT estimation will succeed.")

Total unique pairs after deduplication: 2162
Graph connected: True
All intersections reachable from each other — BT estimation will succeed.


## 5b. Space-filling pair selection

From the ~1,600 candidate pairs, select a focused design that fits within the rater budget.

**Two types of pairs are handled differently:**

- **Bridge pairs (cross-stratum):** all included — mandatory for BT connectivity.
  Each bridge pair is assigned to `BRIDGE_REPLICATION` raters for dropout robustness.

- **Within-stratum pairs:** selected using round-robin over attribute-difference bins
  `(iq_diff, centrum_diff)`. This ensures the selected subset covers the full range of
  "how different are A and B in traffic intensity and location?" rather than clustering
  at one end of the attribute space. Each selected pair assigned to `RATER_REPLICATION` raters.

The number of within-stratum pairs is set by the remaining rater capacity after reserving
slots for bridge pairs.

In [7]:
# Tag each unique pair as bridge (cross-stratum) or within-stratum
bridge_pairs_list = [
    (a, b) for a, b in deduped
    if meta.loc[a, "_stratum"] != meta.loc[b, "_stratum"]
]
within_pairs_list = [
    (a, b) for a, b in deduped
    if meta.loc[a, "_stratum"] == meta.loc[b, "_stratum"]
]

print(f"Candidate pairs: {len(deduped)} total")
print(f"  Bridge (cross-stratum) : {len(bridge_pairs_list)} ({len(bridge_pairs_list)/len(deduped)*100:.1f}%)")
print(f"  Within-stratum         : {len(within_pairs_list)}")

# --- Compute how many within-stratum pairs fit within the rater budget ---
# Each rater sees PAIRS_PER_RATER total, of which N_ANCHOR_PAIRS are anchors shared by all.
# The remaining unique_per_rater slots are filled with replicated non-anchor pairs.
unique_per_rater   = PAIRS_PER_RATER - N_ANCHOR_PAIRS        # 40 non-anchor pairs per rater
total_unique_slots = N_RATERS * unique_per_rater              # total non-anchor rater slots

# Anchors are drawn from bridge pairs — subtract so their slots aren't double-counted.
n_bridge_non_anchor  = max(0, len(bridge_pairs_list) - N_ANCHOR_PAIRS)
bridge_slots_needed  = n_bridge_non_anchor * BRIDGE_REPLICATION
within_slots_left    = total_unique_slots - bridge_slots_needed
n_within_select      = max(0, within_slots_left // RATER_REPLICATION)

print(f"\nRater budget: {N_RATERS} raters × {unique_per_rater} non-anchor slots = {total_unique_slots} total")
print(f"  Bridge non-anchor ({n_bridge_non_anchor}) × {BRIDGE_REPLICATION} reps = {bridge_slots_needed} slots")
print(f"  Within-stratum budget: {within_slots_left} slots ÷ {RATER_REPLICATION} reps → select {n_within_select} pairs")

# --- Apply space-filling selection to within-stratum pairs ---
# Round-robin over (iq_diff, centrum_diff) bins ensures the selected subset
# covers the full attribute-difference space rather than clustering at one end.
selected_within, bin_report = select_space_filling_within(
    within_pairs_list, meta, n_within_select
)
selected_pairs = bridge_pairs_list + selected_within

print(f"\nFinal design: {len(selected_pairs)} unique pairs")
print(f"  Bridge      : {len(bridge_pairs_list)}")
print(f"  Within      : {len(selected_within)} selected from {len(within_pairs_list)} candidates")

# --- Attribute-difference bin coverage report ---
print("\nWithin-stratum coverage by attribute-difference bin:")
print(f"  {'(iq_diff, ct_diff)':>22}  {'selected':>10}  {'available':>12}")
print(f"  {'-'*48}")
for (iq, ct), counts in sorted(bin_report.items(), reverse=True):
    label = f"iq_diff={iq}, ct_diff={ct}"
    print(f"  {label:>22}  {counts['selected']:>10}  {counts['available']:>12}")

# --- Verify connectivity is maintained on the selected subset ---
G_sel = nx.Graph()
G_sel.add_nodes_from(inter_df[id_col].tolist())
G_sel.add_edges_from(selected_pairs)
is_connected_sel = nx.is_connected(G_sel)
print(f"\nSelected design connected: {is_connected_sel}")
if not is_connected_sel:
    n_comp = nx.number_connected_components(G_sel)
    print(f"WARNING: {n_comp} disconnected components — increase BRIDGE_REPLICATION or BRIDGE_PAIRS_PER_LINK")

Candidate pairs: 2162 total
  Bridge (cross-stratum) : 105 (4.9%)
  Within-stratum         : 2057

Rater budget: 16 raters × 40 non-anchor slots = 640 total
  Bridge non-anchor (95) × 3 reps = 285 slots
  Within-stratum budget: 355 slots ÷ 2 reps → select 177 pairs



Final design: 282 unique pairs
  Bridge      : 105
  Within      : 177 selected from 2057 candidates

Within-stratum coverage by attribute-difference bin:
      (iq_diff, ct_diff)    selected     available
  ------------------------------------------------
    iq_diff=3, ct_diff=1          23           112
    iq_diff=3, ct_diff=0          22           205
    iq_diff=2, ct_diff=1          22           409
    iq_diff=2, ct_diff=0          22           471
    iq_diff=1, ct_diff=1          22           382
    iq_diff=1, ct_diff=0          22           307
    iq_diff=0, ct_diff=1          22            28
    iq_diff=0, ct_diff=0          22           143

Selected design connected: False


## 6. Build output dataframe and save

In [8]:
# Rebuild metadata index (also set in load cell; harmless to re-assign here)
meta = inter_df.set_index(id_col)

# Map each selected pair to its type for the output schema
pair_type_map = (
    {frozenset(p): "bridge" for p in bridge_pairs_list}
    | {frozenset(p): "within" for p in selected_within}
)

# Shuffle the selected pair order so each rater sees a mix of strata
shuffle_order = list(range(len(selected_pairs)))
random.Random(RANDOM_SEED).shuffle(shuffle_order)

records = []
for idx in shuffle_order:
    a, b = selected_pairs[idx]
    records.append({
        "pair_id":          idx,
        "intersection_a":   a,
        "image_path_a":     meta.loc[a, img_col],
        "intensity_wvk_a":  meta.loc[a, "intensity_wvk"],
        "stratum_a":        meta.loc[a, "_stratum"],
        "intersection_b":   b,
        "image_path_b":     meta.loc[b, img_col],
        "intensity_wvk_b":  meta.loc[b, "intensity_wvk"],
        "stratum_b":        meta.loc[b, "_stratum"],
        "is_cross_stratum": meta.loc[a, "_stratum"] != meta.loc[b, "_stratum"],
        "pair_type":        pair_type_map.get(frozenset({a, b}), "within"),
    })

# pairs_df is kept as an internal variable — the final merged output is written in section 7
pairs_df = pd.DataFrame(records)

# --- Per-stratum summary (saved separately — small diagnostic file) ---
comp_counts = pd.Series({n: 0 for n in inter_df[id_col].tolist()})
for a, b in selected_pairs:
    comp_counts[a] += 1
    comp_counts[b] += 1

summary_records = []
for stratum, nodes in stratum_nodes.items():
    counts = comp_counts[nodes]
    summary_records.append({
        "stratum":            stratum,
        "n_intersections":    len(nodes),
        "within_pairs":       sum(
            1 for a, b in selected_pairs
            if meta.loc[a, "_stratum"] == stratum
            and meta.loc[b, "_stratum"] == stratum
        ),
        "comp_per_node_min":  int(counts.min()),
        "comp_per_node_mean": round(counts.mean(), 1),
        "comp_per_node_max":  int(counts.max()),
    })
summary_df = pd.DataFrame(summary_records)

os.makedirs(os.path.dirname(SUMMARY_PATH), exist_ok=True)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"Saved summary -> {SUMMARY_PATH}")
print("\nSummary per stratum (selected design only):")
print(summary_df.to_string(index=False))
print(f"\nTotal unique pairs: {len(pairs_df):,}  "
      f"({pairs_df['pair_type'].eq('bridge').sum()} bridge, "
      f"{pairs_df['pair_type'].eq('within').sum()} within-stratum)")
print("pairs_df held in memory — merged output written in section 7")

Saved summary -> C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\bt_pairs_summary.csv

Summary per stratum (selected design only):
         stratum  n_intersections  within_pairs  comp_per_node_min  comp_per_node_mean  comp_per_node_max
          4+_VRI               70           146                  1                 4.7                 40
4+_geen_voorrang               70             5                  0                 0.9                  7
     4+_voorrang               70            11                  0                 0.9                  9
           T_VRI               18            15                  0                 3.6                 24
 T_geen_voorrang               70             0                  0                 0.4                  2
      T_voorrang               70             0                  0                 0.2                  3

Total unique pairs: 282  (105 bridge, 177 within-stratum)
pairs_df he

## 7. Rater assignment

Split the pair schedule across raters, with a shared anchor set seen by everyone.

**Design:**
- **Anchor pairs** (`N_ANCHOR_PAIRS`) — drawn from cross-stratum pairs; shown to all raters.
  Used to check inter-rater reliability and keep scores on a common scale.
- **Unique pairs** — remaining pairs split round-robin across raters so each rater
  gets a balanced mix of strata.

Each rater's schedule is saved to `data/processed/rater_schedules/rater_NN.csv`.
The full `bt_pairs.csv` is unchanged and used as-is for BT model fitting.

In [9]:
unique_per_rater = PAIRS_PER_RATER - N_ANCHOR_PAIRS
total_capacity   = N_RATERS * unique_per_rater

print(f"Design summary:")
print(f"  Unique pairs: {len(pairs_df):,}  ({pairs_df['pair_type'].eq('bridge').sum()} bridge, "
      f"{pairs_df['pair_type'].eq('within').sum()} within-stratum)")
print(f"  Replication : bridge={BRIDGE_REPLICATION}×, within={RATER_REPLICATION}×, anchor=all {N_RATERS} raters")
print(f"  Rater budget: {N_RATERS} × {PAIRS_PER_RATER} = {N_RATERS * PAIRS_PER_RATER} total judgments")

# --- Select anchor pairs from cross-stratum (bridge) pairs ---
rng_anchor     = random.Random(RANDOM_SEED + 1)
cross_pair_ids = list(pairs_df[pairs_df["is_cross_stratum"]]["pair_id"])
anchor_ids     = set(rng_anchor.sample(cross_pair_ids, min(N_ANCHOR_PAIRS, len(cross_pair_ids))))
anchor_df      = pairs_df[pairs_df["pair_id"].isin(anchor_ids)].copy()
anchor_df["is_anchor"] = True
print(f"\nAnchor pairs selected: {len(anchor_df)}")

# --- Non-anchor pairs ---
bridge_non_anchor = pairs_df[
    pairs_df["is_cross_stratum"] & ~pairs_df["pair_id"].isin(anchor_ids)
].copy()
within_non_anchor = pairs_df[~pairs_df["is_cross_stratum"]].copy()

# --- Explicit balanced rater assignment ---
# Assign each pair to exactly R *distinct* raters using load-balancing.
# The old approach (shuffle → round-robin → drop_duplicates) allowed the same
# pair to land on the same rater twice; drop_duplicates then silently shrank
# some sets, producing unequal per-set task counts and crashing the survey framework.
rng_assign = random.Random(RANDOM_SEED + 10)

all_non_anchor_pairs = (
    [(row, BRIDGE_REPLICATION) for _, row in bridge_non_anchor.iterrows()]
    + [(row, RATER_REPLICATION) for _, row in within_non_anchor.iterrows()]
)
rng_assign.shuffle(all_non_anchor_pairs)

rater_load  = [0] * N_RATERS
rater_tasks = [[] for _ in range(N_RATERS)]

for row, n_reps in all_non_anchor_pairs:
    # Pick the n_reps least-loaded raters; random tiebreak keeps the assignment fair
    priority = sorted(range(N_RATERS), key=lambda r: (rater_load[r], rng_assign.random()))
    for r in priority[:n_reps]:
        rater_tasks[r].append(row)
        rater_load[r] += 1

# Trim every rater's list to the minimum count — ensures all sets are exactly equal
n_unique = min(len(t) for t in rater_tasks)
rater_tasks = [t[:n_unique] for t in rater_tasks]
print(f"\nNon-anchor tasks per rater: {n_unique}  "
      f"(total used: {n_unique * N_RATERS} of "
      f"{len(bridge_non_anchor) * BRIDGE_REPLICATION + len(within_non_anchor) * RATER_REPLICATION} replicated)")

cols_out = [
    "pair_id", "intersection_a", "image_path_a", "intensity_wvk_a", "stratum_a",
    "intersection_b", "image_path_b", "intensity_wvk_b", "stratum_b",
    "is_cross_stratum", "pair_type", "is_anchor",
]

def to_vph(x):
    # Convert vehicles/day to rounded vehicles/hour for survey display
    if pd.isna(x):
        return None
    return int(round(float(x) / 24 / 10) * 10)

# --- Save per-rater schedules + build merged design rows ---
os.makedirs(RATER_SCHEDULE_DIR, exist_ok=True)
design_rows = []

for i in range(N_RATERS):
    rater_id = f"rater_{i + 1:02d}"

    # Build per-rater slice directly from pre-assigned rows (no drop_duplicates needed —
    # each pair appears at most once per rater by construction in the assignment above)
    unique_slice = pd.DataFrame(rater_tasks[i])
    unique_slice["is_anchor"] = False
    unique_slice = unique_slice[cols_out]

    schedule = pd.concat([anchor_df[cols_out], unique_slice], ignore_index=True)
    schedule = schedule.sample(frac=1, random_state=RANDOM_SEED + i).reset_index(drop=True)

    schedule.to_csv(os.path.join(RATER_SCHEDULE_DIR, f"{rater_id}.csv"), index=False)

    for _, row in schedule.iterrows():
        design_rows.append({
            "set_id":                  i + 1,
            "pair_id":                 row["pair_id"],
            "is_anchor":               bool(row["is_anchor"]),
            "pair_type":               row["pair_type"],
            "is_cross_stratum":        bool(row["is_cross_stratum"]),
            "intersection_a":          row["intersection_a"],
            "intersection_b":          row["intersection_b"],
            "alt1_att1_streetview":    row["image_path_a"],
            "alt1_att2_intensiteit":   to_vph(row["intensity_wvk_a"]),
            "alt2_att1_streetview":    row["image_path_b"],
            "alt2_att2_intensiteit":   to_vph(row["intensity_wvk_b"]),
            "intensity_wvk_a":         row["intensity_wvk_a"],
            "intensity_wvk_b":         row["intensity_wvk_b"],
            "stratum_a":               row["stratum_a"],
            "stratum_b":               row["stratum_b"],
        })

design_df = pd.DataFrame(design_rows)
design_df.insert(0, "task_id", range(1, len(design_df) + 1))

for col in ["alt1_att2_intensiteit", "alt2_att2_intensiteit"]:
    design_df[col] = pd.array(design_df[col], dtype="Int64")

os.makedirs(os.path.dirname(OUTPUT_DESIGN_PATH), exist_ok=True)
design_df.to_csv(OUTPUT_DESIGN_PATH, index=False)

os.makedirs(OUTPUT_DIR, exist_ok=True)
design_df.to_csv(os.path.join(OUTPUT_DIR, "survey_design.csv"), index=False)

print(f"\nSaved merged design -> {OUTPUT_DESIGN_PATH}")
print(f"Copied to output/   -> {os.path.join(OUTPUT_DIR, 'survey_design.csv')}")
print(f"\n{len(design_df):,} rows | {N_RATERS} sets")

# Verify all sets are exactly equal size — the survey framework requires this
set_sizes = design_df.groupby("set_id").size()
print(f"\nSet sizes: min={set_sizes.min()}, max={set_sizes.max()}  "
      f"(all equal: {set_sizes.min() == set_sizes.max()})")
print(f"Columns: {list(design_df.columns)}")
print(f"\nPer-rater schedules saved to: {RATER_SCHEDULE_DIR}/")

Design summary:
  Unique pairs: 282  (105 bridge, 177 within-stratum)
  Replication : bridge=3×, within=2×, anchor=all 16 raters
  Rater budget: 16 × 50 = 800 total judgments

Anchor pairs selected: 10

Non-anchor tasks per rater: 39  (total used: 624 of 639 replicated)



Saved merged design -> C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\survey_design.csv
Copied to output/   -> C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\output\survey_design.csv

784 rows | 16 sets

Set sizes: min=49, max=49  (all equal: True)
Columns: ['task_id', 'set_id', 'pair_id', 'is_anchor', 'pair_type', 'is_cross_stratum', 'intersection_a', 'intersection_b', 'alt1_att1_streetview', 'alt1_att2_intensiteit', 'alt2_att1_streetview', 'alt2_att2_intensiteit', 'intensity_wvk_a', 'intensity_wvk_b', 'stratum_a', 'stratum_b']

Per-rater schedules saved to: C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\intersections\data\processed\rater_schedules/


# Pas de YAML file aan

In [10]:
import yaml

SURVEY_YAML_PATH = r"C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\core-usage\recipes\recipe-intersection-safety\survey.yaml"

# Derive tasks_per_respondent directly from the generated design: anchors + non-anchor per rater
# n_unique is set in the previous cell and is identical for all raters (trimmed to minimum)
tasks_per_respondent = len(anchor_df) + n_unique

with open(SURVEY_YAML_PATH, "r", encoding="utf-8") as f:
    survey = yaml.safe_load(f)

survey["survey"]["sections"]["body"]["activity_1"]["settings"]["tasks_per_respondent"] = tasks_per_respondent

with open(SURVEY_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(survey, f, allow_unicode=True, sort_keys=False)

print(f"survey.yaml bijgewerkt: tasks_per_respondent = {tasks_per_respondent}")

survey.yaml bijgewerkt: tasks_per_respondent = 49
